# Multi-*e*GO — mg reference simulation with OpenMM

This notebook runs the **molten-globule (mg) reference simulation** required by the multi-eGO workflow.  
The mg simulation produces the trajectory from which contact probabilities are extracted (via `cmdata`) and used to learn the production force field.

## What you need before starting

From the previous steps you should have:

| File | Description |
|------|-------------|
| `topol_mego.top` | GROMACS topology generated by `mego --system … --egos mg` |
| `ffnonbonded.itp` | Non-bonded parameters (included by the topology) |
| `system.gro` | Starting coordinates (from `gmx pdb2gmx` + energy minimisation) |

## Simulation protocol (matches `ff_aa.mdp`)

1. **Energy minimisation** — steepest descents until max force < 500 kJ/mol/nm  
2. **NVT production** — Langevin dynamics, 300 K, 5 fs timestep, 2.5 nm LJ cutoff, no constraints  

> **Note:** This notebook uses [OpenMM](https://openmm.org) and [ParmEd](https://parmed.github.io/ParmEd/) to load and run the GROMACS topology.  The multi-eGO force field uses C6/C12 non-bonded parameters and no electrostatics.  If you encounter loading errors, please open an issue on the [multi-eGO GitHub](https://github.com/multi-ego/multi-eGO).

## 1 · Install dependencies

In [ ]:
!apt update -qq
!apt install -y -qq cmake build-essential libfftw3-dev wget ninja-build gromacs

In [ ]:
#!git clone --depth 1 -b release-2023 https://github.com/multi-ego/gromacs.git
#%cd gromacs
#%mkdir build
#%cd build
#!cmake -G Ninja .. -DBUILD_TESTING=OFF -DGMX_INSTALL_LEGACY_API=ON -DGMX_IMD=OFF > /dev/null
#!ninja -j$(nproc)
#!ninja install
#!ln -s /usr/local/gromacs/bin/gmx /usr/bin/gmx

In [9]:
# Install ParmEd, and MDTraj (for trajectory conversion)
!pip install -q parmed mdtraj matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 66.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 70.6 MB/s eta 0:00:00


## 2 · Upload input files

In [10]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(
    accept='.pdb',
    multiple=False,
    description='Upload PDB'
)

display(upload)

FileUpload(value={}, accept='.pdb', description='Upload PDB')

In [11]:
import os

if upload.value:
    file_info = list(upload.value.values())[0]
    filename = file_info['metadata']['name']

    with open(filename, "wb") as f:
        f.write(file_info['content'])

    print("Saved:", filename)

Saved: 2efv_Y16W.pdb


In [7]:
!git clone --depth 1 https://github.com/multi-ego/multi-ego.git > /dev/null
%cd multi-ego
!pip install -e . > /dev/null
%cd ..
import os
os.environ["GMXLIB"] = os.path.abspath("multi-ego")
print(os.environ["GMXLIB"])

Cloning into 'multi-ego'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (284/284), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 284 (delta 53), reused 179 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (284/284), 73.70 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (53/53), done.
Updating files: 100% (240/240), done.
/content/multi-ego
Obtaining file:///content/multi-ego
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mego (pyproject.toml) ... done
  Created wheel for mego: filename=mego-0.6.0-0.editable-py3-none-any.whl size=13885 sha256=6df6c000a4ce90defb4f8eb56ca35ccedcfd3ec7518a2f0129d90478c9726a33
  Stored in directory: /tmp/pip-ephem-wheel-cache-z5d0bqst/wheels/1e/c2/5c/c20b71c9adef1c194e93e2db264aba24f3dd0498dabf2892db
Successfully b

In [8]:
%ls

gromacs/  multi-ego/  sample_data/


In [16]:
!printf "1\n" | gmx pdb2gmx -f {filename} -o processed.gro -water none

             :-) GROMACS - gmx pdb2gmx, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            

In [18]:
!gmx editconf -f processed.gro -o boxed.gro -c -bt cubic -d 10.0
#

             :-) GROMACS - gmx editconf, 2021.4-Ubuntu-2021.4-2 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff           

In [20]:
replacements = {
    "_TEMP_": "300",
    "_SET_": "1000000",
}

with open("multi-ego/mdps/ff_aa.mdp") as f:
    content = f.read()

for k, v in replacements.items():
    content = content.replace(k, str(v))

with open("ff_aa.mdp", "w") as f:
    f.write(content)

In [23]:
!mkdir -p multi-ego/inputs/prova
!cp topol.top multi-ego/inputs/prova/topol.top
!mego --system prova --egos mg --inputs_dir multi-ego/inputs/


-----------------------------===========+===+++++++++++++********#########################
-----------------------------=========++++**##########**********##########################
-----------------------------=-==+**#%%%%####*****#####%%%@%#########%%%%%################
----------------------=---====*#%@@@%%%%##**#******+****##%%@@@@@%%%%%%%##################
-------------------------==+#%@@@%%%###***+*+++=+*+*+++***##%@@@@@@@@@@%#%################
-----------------------==+%@@%%%###***+++========++=+++++**####%%@@@@@@@@%################
---------------------=+#%@@%###****+++++=============++=++++***###%@@@@@@@%###############
--------------------=*%%@@%###***+++====================+++++****###%@@@@@@%##############
------------------=*%@@@@%#***+++==========--============++++******##%%@@@@@@%############
-----------------=#@@@@@%#***++========-===--===--========+++++++****##%@@@@@@%#####%%%%%#
----------------+@@@@@%##**+++=====-==-=------------=======++++++++***##%%@@@@@###%%%%%%%

In [ ]:
!gmx_mpi grompp -f ff_aa -c boxed.gro -p topol_mego.top -o run.tpr
#

## 3 · Simulation parameters

Edit the values in this cell to match your system.

## 4 · Load topology and build OpenMM system

## 5 · Energy minimisation

## 6 · NVT production run

The trajectory is saved as `trajectory.dcd` and converted to `.xtc` at the end for compatibility with `cmdata`.

## 7 · Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LOG_FILE, sep='\t', comment='#')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

time_ns = df['Time (ps)'] / 1000

axes[0].plot(time_ns, df['Potential Energy (kJ/mole)'], color='steelblue', lw=0.8)
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Potential energy (kJ/mol)')
axes[0].set_title('Potential Energy')
axes[0].grid(alpha=0.3)

axes[1].plot(time_ns, df['Temperature (K)'], color='tomato', lw=0.8)
axes[1].axhline(TEMPERATURE_K, color='k', ls='--', lw=1, label=f'{TEMPERATURE_K} K')
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Temperature (K)')
axes[1].set_title('Temperature')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('energy_plot.png', dpi=150)
plt.show()
print(f"Mean temperature : {df['Temperature (K)'].mean():.1f} ± {df['Temperature (K)'].std():.1f} K")
print(f"Mean pot. energy : {df['Potential Energy (kJ/mole)'].mean():.1f} kJ/mol")

## 8 · Convert trajectory to XTC (for `cmdata`)

`cmdata` expects a GROMACS `.xtc` trajectory.  MDTraj converts the `.dcd` output here.

In [ ]:
import mdtraj as md

print("Loading DCD trajectory...")
traj = md.load(TRAJ_FILE, top='final.pdb')
print(f"  Frames : {traj.n_frames}")
print(f"  Atoms  : {traj.n_atoms}")

xtc_file = TRAJ_FILE.replace('.dcd', '.xtc')
traj.save_xtc(xtc_file)
print(f"Trajectory saved as {xtc_file} ✓")

## 9 · Download output files

Download the trajectory (`.xtc`) and the energy log.  You will need the `.xtc` file together with a GROMACS `.tpr` run-input file to extract contact matrices with `cmdata`.

In [ ]:
from google.colab import files as colab_files
import os

outputs = [xtc_file, LOG_FILE, 'energy_plot.png', 'final.pdb', CHECKPOINT_FILE]
for f in outputs:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        colab_files.download(f)

## Next steps

With the trajectory in hand:

1. Run `cmdata` on the `.xtc` trajectory (requires a `.tpr` file from GROMACS):
   ```bash
   cmdata -f trajectory.xtc -s topol.tpr
   ```

2. Convert the histograms to contact matrices:
   ```bash
   python tools/make_mat/make_mat.py --histo histo/ ...
   ```

3. Generate the production force field:
   ```bash
   mego --config inputs/SYSTEM/config.yml
   ```

See the [multi-eGO documentation](https://github.com/multi-ego/multi-eGO) for the full workflow.